In [6]:
      #!/bin/bash
!kaggle datasets download gauravdhamane/gwa-bitbrains

Dataset URL: https://www.kaggle.com/datasets/gauravdhamane/gwa-bitbrains
License(s): other
gwa-bitbrains.zip: Skipping, found more recently modified local copy (use --force to force download)


In [7]:
import os
import pandas as pd

# 1. Unzip the dataset quietly (-q) into a new folder
!unzip -q /content/gwa-bitbrains.zip -d /content/bitbrains_data

# 2. Get a list of all the extracted CSV files
folder_path = '/content/bitbrains_data'
# Note: The unzip might create a subfolder, so we search for the first .csv
all_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith('.csv'):
            all_files.append(os.path.join(root, file))

print(f"Total CSV files found: {len(all_files)}")

# 3. Load just ONE file to verify the structure and delimiters
# Bitbrains often uses a semicolon and a tab as a separator
first_file = all_files[0]
print(f"\nLoading sample file: {first_file}")

df = pd.read_csv(first_file, sep=';\t', engine='python')

# 4. Print the columns and the first 5 rows
print("\n--- Column Names ---")
print(df.columns.tolist())

print("\n--- First 5 Rows ---")
print(df.head())

replace /content/bitbrains_data/fastStorage/2013-8/1.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: Total CSV files found: 1250

Loading sample file: /content/bitbrains_data/fastStorage/2013-8/755.csv

--- Column Names ---
['Timestamp [ms]', 'CPU cores', 'CPU capacity provisioned [MHZ]', 'CPU usage [MHZ]', 'CPU usage [%]', 'Memory capacity provisioned [KB]', 'Memory usage [KB]', 'Disk read throughput [KB/s]', 'Disk write throughput [KB/s]', 'Network received throughput [KB/s]', 'Network transmitted throughput [KB/s]']

--- First 5 Rows ---
   Timestamp [ms]  CPU cores  CPU capacity provisioned [MHZ]  CPU usage [MHZ]  \
0      1376314846          1                      2599.99945         0.000000   
1      1376315146          1                      2599.99945         5.199999   
2      1376315446          1                      2599.99945         0.000000   
3      1376315746          1                      2599.99945         0.000000   
4      1376316046          1                      259

In [8]:
df = pd.read_csv('/content/bitbrains_data/fastStorage/2013-8/1.csv')

In [9]:
len(df)

8634

In [10]:
from google.colab import drive
import os

# 1. Mount the drive
drive.mount('/content/drive')

# 2. Define your project path
PROJECT_NAME = "Bitbrains_LSTM_Predictive_NEW_Scaler"
DRIVE_PATH = f"/content/drive/MyDrive/{PROJECT_NAME}"

if not os.path.exists(DRIVE_PATH):
    os.makedirs(DRIVE_PATH)
    print(f"Project folder created at: {DRIVE_PATH}")
else:
    print(f"Project folder already exists at: {DRIVE_PATH}")

Mounted at /content/drive
Project folder already exists at: /content/drive/MyDrive/Bitbrains_LSTM_Predictive_NEW_Scaler


In [11]:
def create_tf_dataset(file_list):
    dataset = tf.data.Dataset.from_generator(
        lambda: vm_data_generator(file_list),
        output_signature=(
            tf.TensorSpec(shape=(WINDOW_SIZE, 5), dtype=tf.float32),  # 4 → 5
            tf.TensorSpec(shape=(3,),             dtype=tf.float32)   # 2 → 3
        )
    )
    return (dataset
            .batch(BATCH_SIZE)
            .repeat()
            .prefetch(2))

In [12]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

# 1. File-Level Train/Validation Split
random.seed(42) # Ensure reproducible splits
random.shuffle(all_files)

TRAIN_FILES = all_files[:1000]
VAL_FILES = all_files[1000:]

print(f"Training on {len(TRAIN_FILES)} VMs.")
print(f"Validating on {len(VAL_FILES)} unseen VMs.")

WINDOW_SIZE = 12
BATCH_SIZE = 4096 # Massive batch size for fast GPU processing




train_dataset = create_tf_dataset(TRAIN_FILES)
val_dataset = create_tf_dataset(VAL_FILES)

print("Data Pipeline completely built and ready to stream to GPU!")

Training on 1000 VMs.
Validating on 250 unseen VMs.
Data Pipeline completely built and ready to stream to GPU!


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Input , LSTM
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
import tensorflow as tf

# Define the GRU
model = Sequential([
    Input(shape=(WINDOW_SIZE, 5)),

    LSTM(128, return_sequences=True),
    Dropout(0.2),

    LSTM(64, return_sequences=False),
    Dropout(0.2),

    Dense(32, activation='relu'),
    Dense(3, activation='linear')    # 2 → 3
])

# Huber loss is highly resilient to massive server anomalies
def accuracy_metric(y_true, y_pred):
    # Counts a prediction as "accurate" if it's within 10% (0.10) of the actual value
    diff = tf.abs(y_true - y_pred)
    return tf.reduce_mean(tf.cast(diff < 0.10, tf.float32))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=['mae', accuracy_metric] # Now "accuracy_metric" will print every step!
)

# 4. The Safety Nets (Callbacks)
# Stop training if the validation loss stops improving, and save the absolute best version
# 1. Path for saving models: e.g., lstm_epoch_01.keras, lstm_epoch_02.keras
checkpoint_path = DRIVE_PATH + "/lstm_epoch_{epoch:02d}.keras"  # .h5 → .keras

# 2. Path for saving the training log (CSV)
log_path = DRIVE_PATH + "/lstm_training_log.csv"

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=False,
        verbose=1
    ),

    CSVLogger(log_path, append=True)
]

print("Callbacks configured to save all data to Google Drive.")
print("Architecture built. Ready to train.")

Callbacks configured to save all data to Google Drive.
Architecture built. Ready to train.


In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 12, 128)        │        68,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 120,195 (469.51 KB)

 Trainable params: 120,195 (469.51 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
import gc

def vm_data_generator(file_list):
    scaler = MinMaxScaler(feature_range=(0, 1))

    # shuffle files each epoch so model sees different order
    shuffled_files = file_list.copy()
    random.shuffle(shuffled_files)

    for file_path in shuffled_files:
        try:
            df = pd.read_csv(file_path, sep=';\t', engine='python')
            if len(df) <= WINDOW_SIZE:
                continue

            df['datetime']   = pd.to_datetime(df['Timestamp [ms]'], unit='s')
            df['min_of_day'] = df['datetime'].dt.hour * 60 + df['datetime'].dt.minute

            cpu   = df['CPU usage [%]'].values
            ram   = df['Memory usage [KB]'].values
            diskw = df['Disk write throughput [KB/s]'].values  # ← new
            t_sin = np.sin(2 * np.pi * df['min_of_day'] / 1440.0).values
            t_cos = np.cos(2 * np.pi * df['min_of_day'] / 1440.0).values

            data        = np.column_stack((cpu, ram, diskw, t_sin, t_cos))  # 4 → 5
            scaled_data = scaler.fit_transform(data)

            for i in range(len(scaled_data) - WINDOW_SIZE):
                X = scaled_data[i : i + WINDOW_SIZE]
                y = scaled_data[i + WINDOW_SIZE, 0:3]  # CPU, RAM, Disk write
                yield X, y

            # release memory immediately after each file
            del df, data, scaled_data, cpu, ram, diskw, t_sin, t_cos
            gc.collect()

        except Exception as e:
            continue

In [16]:
# Calculate exact steps automatically
print("Calculating total number of individual samples from the generators. This might take several minutes, please be patient...")

# Initialize counts
train_sample_count = 0
val_sample_count = 0

# # Count samples from the training data generator
# for _ in vm_data_generator(TRAIN_FILES):
#     train_sample_count += 1
print(f"Finished counting training samples: {train_sample_count}")

# # Count samples from the validation data generator
# for _ in vm_data_generator(VAL_FILES):
#     val_sample_count += 1
print(f"Finished counting validation samples: {val_sample_count}")

# Calculate steps per epoch
# Use integer division as steps must be integers
STEPS_PER_EPOCH = 8839374 // BATCH_SIZE
VALIDATION_STEPS = 2307756 // BATCH_SIZE

print(f"\n--- Calculated Steps ---")
print(f"Total training samples: {train_sample_count}")
print(f"Total validation samples: {val_sample_count}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"STEPS_PER_EPOCH (batches): {STEPS_PER_EPOCH}")
print(f"VALIDATION_STEPS (batches): {VALIDATION_STEPS}")

Calculating total number of individual samples from the generators. This might take several minutes, please be patient...
Finished counting training samples: 0
Finished counting validation samples: 0

--- Calculated Steps ---
Total training samples: 0
Total validation samples: 0
BATCH_SIZE: 4096
STEPS_PER_EPOCH (batches): 2158
VALIDATION_STEPS (batches): 563


In [ ]:
EPOCHS = 50
# STEPS_PER_EPOCH and VALIDATION_STEPS will be updated by the previous cell.
# The values below are placeholders and will be overwritten during execution.
# They are kept here for initial setup or if the calculation cell is skipped.
# STEPS_PER_EPOCH = 3907
# VALIDATION_STEPS = 977 # (250 files * 8000 / 512)

print(f"🚀 Starting Training. All progress will be saved to: {DRIVE_PATH}")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,

    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
    callbacks=callbacks
)

🚀 Starting Training. All progress will be saved to: /content/drive/MyDrive/Bitbrains_LSTM_Predictive_NEW_Scaler
Epoch 1/50
2158/2158 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy_metric: 0.8407 - loss: 0.0057 - mae: 0.0575
Epoch 1: saving model to /content/drive/MyDrive/Bitbrains_LSTM_Predictive_NEW_Scaler/lstm_epoch_01.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Bitbrains_LSTM_Predictive_NEW_Scaler/lstm_epoch_01.keras
2158/2158 ━━━━━━━━━━━━━━━━━━━━ 3424s 2s/step - accuracy_metric: 0.8637 - loss: 0.0044 - mae: 0.0499 - val_accuracy_metric: 0.8901 - val_loss: 0.0035 - val_mae: 0.0424
Epoch 2/50
1136/2158 ━━━━━━━━━━━━━━━━━━━━ 21:24 1s/step - accuracy_metric: 0.8761 - loss: 0.0036 - mae: 0.0455

In [ ]:
import os
import shutil
import tensorflow as tf

EPOCHS    = 10
LAST_EPOCH = 1   # Changed to 1, as training only completed epoch 1

# =========================================================
# COPY CHECKPOINT FROM DRIVE TO LOCAL SSD
# =========================================================
MODEL_NAME       = f"lstm_epoch_01.keras"
DRIVE_MODEL_PATH = os.path.join(DRIVE_PATH, MODEL_NAME)
LOCAL_MODEL_DIR  = "/content/local_models"
LOCAL_MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, MODEL_NAME)

os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)
print("Copying checkpoint to local storage...")
shutil.copy(DRIVE_MODEL_PATH, LOCAL_MODEL_PATH)

# =========================================================
# LOAD CHECKPOINT
# =========================================================
print(f"Loading model from epoch {LAST_EPOCH}...")

def accuracy_metric(y_true, y_pred):
    diff = tf.abs(y_true - y_pred)
    return tf.reduce_mean(tf.cast(diff < 0.10, tf.float32))

model = tf.keras.models.load_model(
    LOCAL_MODEL_PATH,
    custom_objects={'accuracy_metric': accuracy_metric},
    compile=False
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=['mae', accuracy_metric]
)

print("Weights restored and model recompiled!")

# =========================================================
# RE-INITIALIZE DATASETS TO REFLECT LATEST GENERATOR DEFS
# =========================================================
train_dataset = create_tf_dataset(TRAIN_FILES)
val_dataset = create_tf_dataset(VAL_FILES)

# =========================================================
# RESUME TRAINING — datasets already have .batch().repeat().prefetch()
# =========================================================
print("Resuming training...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    initial_epoch=LAST_EPOCH,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
    callbacks=callbacks,
    verbose=1
)

print("Training complete!")

In [ ]:
import os
import shutil
import subprocess
import numpy as np

# Install first
subprocess.run(["pip", "install", "onnxruntime", "tf2onnx", "onnx", "-q"])

import tensorflow as tf
import onnxruntime as ort

DRIVE_PATH   = "/content/drive/MyDrive/Bitbrains_LSTM_Predictive_Scaler"
KERAS_FILE   = f"{DRIVE_PATH}/lstm_epoch_04.keras"
LOCAL_KERAS  = "/content/lstm_epoch_04.keras"
SAVED_MODEL  = "/content/lstm_saved_model"
LOCAL_ONNX   = "/content/lstm_autoscaler.onnx"
ONNX_FILE    = f"{DRIVE_PATH}/lstm_autoscaler.onnx"

def accuracy_metric(y_true, y_pred):
    diff = tf.abs(y_true - y_pred)
    return tf.reduce_mean(tf.cast(diff < 0.10, tf.float32))

# ── Load keras model ──────────────────────────────────────────────
shutil.copy(KERAS_FILE, LOCAL_KERAS)
model = tf.keras.models.load_model(
    LOCAL_KERAS,
    custom_objects={'accuracy_metric': accuracy_metric},
    compile=False
)
print("Loaded | Input:", model.input_shape, "Output:", model.output_shape)

# ── Save as SavedModel on CPU ─────────────────────────────────────
if os.path.exists(SAVED_MODEL):
    shutil.rmtree(SAVED_MODEL)
tf.saved_model.save(model, SAVED_MODEL)
print("SavedModel written.")

# ── Convert via subprocess — this is what works ───────────────────
result = subprocess.run([
    "python3", "-m", "tf2onnx.convert",
    "--saved-model", SAVED_MODEL,
    "--output", LOCAL_ONNX,
    "--opset", "13"
], capture_output=True, text=True)

print(result.stderr[-800:])  # shows conversion log

if result.returncode != 0:
    print("CONVERSION FAILED")
    print(result.stderr)
else:
    print("ONNX exported successfully!")

    # ── Verify ────────────────────────────────────────────────────
    dummy     = np.random.rand(2, 60, 5).astype(np.float32)
    sess      = ort.InferenceSession(LOCAL_ONNX, providers=['CPUExecutionProvider'])
    iname     = sess.get_inputs()[0].name
    out_onnx  = sess.run(None, {iname: dummy})[0]
    out_keras = model.predict(dummy, verbose=0)

    print(f"ONNX  : {out_onnx[0]}")
    print(f"Keras : {out_keras[0]}")
    print(f"Diff  : {np.max(np.abs(out_onnx - out_keras)):.6f}")

    # ── Save to Drive ─────────────────────────────────────────────
    shutil.copy(LOCAL_ONNX, ONNX_FILE)
    print("Saved to Drive — download lstm_autoscaler.onnx!")